In [1]:
import json
from utils_analysis_var_decomp import between_within_var_breakdown

In [ ]:
# def generate_features_of_sampled_idxs(
#     df_sampling: pd.DataFrame, df_candidates: pd.DataFrame, tensor_candidates: Tensor
# ):
#     assert "indices" in df_sampling.columns
#     assert "baseline_logprob" in df_candidates.columns
#     assert "source_delfy" in df_candidates.columns
#     assert "target_delfy" in df_candidates.columns

#     df_sampling["baseline_logprob_min"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["baseline_logprob"].min()  # type: ignore
#     )
#     df_sampling["baseline_logprob_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["baseline_logprob"].mean()  # type: ignore
#     )
#     df_sampling["source_delfy_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["source_delfy"].mean()  # type: ignore
#     )
#     df_sampling["target_delfy_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["target_delfy"].mean()  # type: ignore
#     )
#     df_sampling["baseline_informativeness_bald_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["baseline_informativeness_bald"].mean()  # type: ignore
#     )
#     df_sampling["baseline_informativeness_ate_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["baseline_informativeness_ate"].mean()  # type: ignore
#     )
#     df_sampling["baseline_informativeness_ls_mean"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: df_candidates.loc[lst]["baseline_informativeness_ls"].mean()  # type: ignore
#     )
#     df_sampling["baseline_avg_l2_from_center"] = df_sampling["indices"].apply(  # type: ignore
#         lambda lst: compute_avg_l2_dist_from_center(tensor_candidates[lst])  # type: ignore
#     )
#     return df_sampling

# df_sampling = generate_features_of_sampled_idxs(
#     df_sampling=df_sampling,
#     df_candidates=df_candidates,
#     tensor_candidates=hidden_states,
# )

In [2]:
def open_folder(
    task_name: str,
    model_name: str,
    num_samples: int,
    num_shuffles: int,
    num_datapoints: int,
):
    base_path = "/home/mila/f/floresl/active-learning-assumptions/outputs"
    return json.load(
        open(
            f"{base_path}/{task_name}/{model_name}_{num_samples}_samples_{num_datapoints}_datapoints_{num_shuffles}_shuffles_200_eps_bs_8_lr_5e-05.json",
            "r",
        )
    )

In [22]:
for task in ["eng_afr", "eng_deu", "eng_fil"]:
    for model_name in ["t5", "llama", "gemma"]:
        try:
            var_decomp_result = open_folder(
                task_name=task,
                model_name=model_name,
                num_samples=10,
                num_shuffles=10,
                num_datapoints=500,
            )
        except FileNotFoundError:
            print(f"File not found for Task: {task}, Model: {model_name}")
            continue
        chrfs_by_trial = [
            [shuffle["eval_chrf"] for shuffle in trial["eval_metrics"]]
            for trial in var_decomp_result
        ]
        var_decomp = between_within_var_breakdown(chrfs_by_trial, G=10, N=10)
        print(
            f"Task: {task}, Model: {model_name}, % Within: {var_decomp['ss_within'] / var_decomp['ss_total']:.2f}"
        )

File not found for Task: eng_afr, Model: t5
Task: eng_afr, Model: llama, % Within: 0.92
Task: eng_afr, Model: gemma, % Within: 0.84
Task: eng_deu, Model: t5, % Within: 0.81
Task: eng_deu, Model: llama, % Within: 0.94
Task: eng_deu, Model: gemma, % Within: 0.65
Task: eng_fil, Model: t5, % Within: 0.93
Task: eng_fil, Model: llama, % Within: 0.80
Task: eng_fil, Model: gemma, % Within: 0.80


{'ss_total': 48.49111544202697,
 'ss_within': 40.53924007365252,
 'ss_between': 7.951875368374344,
 'ss_total_plus_between': 48.49111544202687}